In [1]:
import pickle 
import numpy as np 
import itertools
import pandas as pd



# Generate conditions for symmetric distractor test.

Test in style of Kidd / Marrone et al. (2008) / Srinivasan et al. (2016) experiments 

Target at 0 aziuth    
Distractors symmetrically presented at +/- 0, 5, 10, 20, and 40 degrees azimuth    
All signals at 0 elevation     
Left and right distractor level summed and set to 60dB SPL
- equal power at left and right for model first pass
- human experiments roved L/R balance slightly

Target set against distractor at desired SNR 

### Want to test effect of reverberation in this setting 
Use rooms at `/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/` which are versions of the speaker array room with alternative reverb profiles.

## This Notebook
Generate location x snr conditions to get model performance and thresholds, per room 

### Load manifest of rooms

In [2]:
room_manifest = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/manifest_room.pdpkl')

In [4]:
room_manifest

,head_azim,head_pos_xyz,index_room,is_outdoor,material_x0,material_x1,material_y0,material_y1,material_z0,material_z1,room_dim_xyz,room_materials
0,0,"(2.5, 2.5, 2.0)",0,False,"(Anechoic,)","(Anechoic,)","(Anechoic,)","(Anechoic,)","(Anechoic,)","(Anechoic,)","(5.0, 5.0, 4.0)","(26, 26, 26, 26, 26, 26)"
1,0,"(2.5, 2.5, 2.0)",1,False,"(Wood panelling on glass fiber blanket,)","(Wood panelling on glass fiber blanket,)","(Wood panelling on glass fiber blanket,)","(Wood panelling on glass fiber blanket,)","(Carpet on foam rubber padding,)","(Highly absorptive panels, 1"", 16"" below ceili...","(5.0, 5.0, 4.0)","(11, 11, 11, 11, 15, 20)"
2,0,"(2.5, 2.5, 2.0)",2,False,"(Brick,)","(Brick,)","(Brick,)","(Brick,)","(Wood parquet on concrete,)","(Plaster, gypsum, or lime on lath,)","(5.0, 5.0, 4.0)","(1, 1, 1, 1, 12, 16)"
3,0,"(2.5, 2.5, 2.0)",3,False,"(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Linoleum,)","(Acoustic tiles, 0.625"", 16"" below ceiling,)","(5.0, 5.0, 4.0)","(9, 9, 9, 9, 13, 17)"
4,0,"(2.5, 2.5, 2.0)",4,False,"(Concrete, painted,)","(Concrete, painted,)","(Concrete, painted,)","(Concrete, painted,)","(Linoleum,)","(Acoustic tiles, 0.625"", 16"" below ceiling,)","(5.0, 5.0, 4.0)","(2, 2, 2, 2, 13, 17)"
5,0,"(2.3, 3.6, 0.9)",5,False,"(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Concrete, painted,)","(Linoleum,)","(Acoustic tiles, 0.625"", 16"" below ceiling,)","(4.66, 5.9, 2.48)","(9, 9, 9, 2, 13, 17)"
6,0,"(2.33, 2.95, 0.9)",6,False,"(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Concrete, painted,)","(Linoleum,)","(Acoustic tiles, 0.625"", 16"" below ceiling,)","(4.66, 5.9, 2.48)","(9, 9, 9, 2, 13, 17)"
7,0,"(2.3, 3.6, 0.9)",7,False,"(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Linoleum,)","(Acoustic tiles, 0.625"", 16"" below ceiling,)","(4.66, 5.9, 2.48)","(9, 9, 9, 9, 13, 17)"
8,0,"(2.33, 2.95, 0.9)",8,False,"(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Fiberglass wall treatment, 1 in,)","(Linoleum,)","(Acoustic tiles, 0.625"", 16"" below ceiling,)","(4.66, 5.9, 2.48)","(9, 9, 9, 9, 13, 17)"


In [5]:
room_ir_df = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/manifest_brir.pdpkl')
elevations = room_ir_df[(room_ir_df.src_dist == 1.4) & (room_ir_df.index_room == 0)].src_elev.unique() # np.arange(-20, 41, 10)
azimuths = room_ir_df[(room_ir_df.src_dist == 1.4) & (room_ir_df.index_room == 0)].src_azim.unique() # np.arange(-20, 41, 10)
# remove 0 elevation
print(azimuths)
print(elevations)


[  0   5  10  15  20  25  30  35  40  45  50  55  60  65  70  75  80  85
  90  95 100 105 110 115 120 125 130 135 140 145 150 155 160 165 170 175
 180 185 190 195 200 205 210 215 220 225 230 235 240 245 250 255 260 265
 270 275 280 285 290 295 300 305 310 315 320 325 330 335 340 345 350 355]
[-40 -30 -20 -10   0  10  20  30  40  50  60]


### Get range of conditions that match human tests (ours and published)

In [5]:
np.linspace(6,-12,num=7)

array([  6.,   3.,   0.,  -3.,  -6.,  -9., -12.])

'/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/room0001.hdf5'

In [12]:
# First add conditions for confusion matrix 
target_azim = 0
elevation = 0 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 
snrs = np.linspace(6,-15,num=8)
print(snrs)
room_ixs = room_manifest.index_room.unique()
print(room_ixs)

def get_room_str(room_ix):
    return f"/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/room{room_ix:04}.hdf5"

def get_room_meta_dict(room_ix):
    manifest_path = "/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/manifest_brir.pdpkl"
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts
all_cond_dict = {ix:{'target_loc': (target_azim, elevation),
                      'distract_loc': [dist_az, elevation],
                      'snr': snr,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (dist_az, snr, room_ix) in
              enumerate(itertools.product(*[distractor_azims, snrs, room_ixs]))} 

n_all_cond_dict = len(all_cond_dict)
print(n_all_cond_dict)

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

[  6.   3.   0.  -3.  -6.  -9. -12. -15.]
[0 1 2 3 4 5 6 7 8]
576


In [16]:
# all_cond_dict

In [13]:
# all_conds
# all conditions 
with open('binaural_test_manifests/symmetric_distractor_conditions_neg_15_to_6_dBSNR_eval_rooms.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)

In [12]:
# # all_conds
# # all conditions 
# with open('binaural_test_manifests/symmetric_distractor_conditions_neg_12_to_6_dBSNR.pkl', 'wb') as f:
#     pickle.dump(all_cond_dict, f)

In [9]:
# # all_conds
# # all conditions 
# with open('binaural_test_manifests/symmetric_distractor_conditions_neg_12_to_6_dBSNR.pkl', 'rb') as f:
#     cond_dict = pickle.load(f)

In [8]:
# First add conditions for confusion matrix 
target_azim = 0
elevation = 0 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 
# snrs = np.linspace(3,-15,num=7)
# create dict of trial dicts
all_cond_dict = {ix:{'target_loc': (target_azim, elevation),
                      'distract_loc': [dist_az, elevation],
                      'snr': 6} for ix, dist_az in
              enumerate(distractor_azims, )} 

n_all_cond_dict = len(all_cond_dict)
print(n_all_cond_dict)

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

8


# sanity check old rooms 

In [2]:
room_manifest = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/manifest_room.pdpkl')

In [3]:
room_ir_df = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/manifest_brir.pdpkl')
elevations = room_ir_df[(room_ir_df.src_dist == 1.4) & (room_ir_df.index_room == 0)].src_elev.unique() # np.arange(-20, 41, 10)
azimuths = room_ir_df[(room_ir_df.src_dist == 1.4) & (room_ir_df.index_room == 0)].src_azim.unique() # np.arange(-20, 41, 10)
# remove 0 elevation
print(azimuths)
print(elevations)


[  0   5  10  15  20  25  30  35  40  45  50  55  60  65  70  75  80  85
  90  95 100 105 110 115 120 125 130 135 140 145 150 155 160 165 170 175
 180 185 190 195 200 205 210 215 220 225 230 235 240 245 250 255 260 265
 270 275 280 285 290 295 300 305 310 315 320 325 330 335 340 345 350 355]
[-20 -10   0  10  20  30  40]


In [4]:
# First add conditions for confusion matrix 
target_azim = 0
elevation = 0 
distractor_azims = [0, 5, 10, 20, 30, 40, 60, 90] # Add 30, 60, 90 for comparison to our human experiments - test script handles l/r positioning 
snrs = np.linspace(6,-15,num=8)
print(snrs)
room_ixs = [0]
print(room_ixs)

def get_room_str(room_ix):
    return f"/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/room{room_ix:04}.hdf5"

def get_room_meta_dict(room_ix):
    manifest_path = "/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval/manifest_brir.pdpkl"
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts
all_cond_dict = {ix:{'target_loc': (target_azim, elevation),
                      'distract_loc': [dist_az, elevation],
                      'snr': snr,
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, (dist_az, snr, room_ix) in
              enumerate(itertools.product(*[distractor_azims, snrs, room_ixs]))} 

n_all_cond_dict = len(all_cond_dict)
print(n_all_cond_dict)

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

[  6.   3.   0.  -3.  -6.  -9. -12. -15.]
[0]
64


In [5]:
# all_conds
# all conditions 
with open('binaural_test_manifests/symmetric_distractor_conditions_neg_15_to_6_dBSNR_mit_room.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)